# Load Libraries

In [ ]:
#load libraries
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
print('matplotlib:', matplotlib.__version__)

matplotlib: 3.10.8


# Load Data

In [ ]:
#load data
df_untidy = pd.read_csv('mutant_moneyball.csv')
print('Shape:', df_untidy.shape)
print('\nColumn names:')
for col in df_untidy.columns:
    print(' ', col)

df_untidy.head()

Shape: (26, 17)

Column names:
  Member
  TotalValue60s_heritage
  TotalValue70s_heritage
  TotalValue80s_heritage
  TotalValue90s_heritage
  TotalValue60s_ebay
  TotalValue70s_ebay
  TotalValue80s_ebay
  TotalValue90s_ebay
  TotalValue60s_wiz
  TotalValue70s_wiz
  TotalValue80s_wiz
  TotalValue90s_wiz
  TotalValue60s_oStreet
  TotalValue70s_oStreet
  TotalValue80s_oStreet
  TotalValue90s_oStreet


,Member,TotalValue60s_heritage,TotalValue70s_heritage,TotalValue80s_heritage,TotalValue90s_heritage,TotalValue60s_ebay,TotalValue70s_ebay,TotalValue80s_ebay,TotalValue90s_ebay,TotalValue60s_wiz,TotalValue70s_wiz,TotalValue80s_wiz,TotalValue90s_wiz,TotalValue60s_oStreet,TotalValue70s_oStreet,TotalValue80s_oStreet,TotalValue90s_oStreet
0,warrenWorthington,929056.0,154585.0,23957.0,960.0,23335.0,3362.0,583.0,97.0,"$7,913.00","$1,105.00",$226.00,$65.75,"$68,160.00","$7,360.00",$975.00,$123.00
1,hankMcCoy,929776.0,20705.0,6631.0,881.0,23377.0,1224.0,289.0,82.0,"$7,953.00",$851.00,$89.00,$38.50,"$68,390.00","$5,260.00",$431.00,$81.00
2,scottSummers,933616.0,188635.0,29240.0,739.0,23420.0,5431.0,1031.0,82.0,"$7,993.00","$1,979.00",$438.00,$39.25,"$68,590.00","$11,675.00","$1,427.00",$74.00
3,bobbyDrake,929776.0,154585.0,1514.0,874.0,23377.0,3362.0,70.0,93.0,"$7,953.00","$1,105.00",$48.00,$62.00,"$68,390.00","$7,360.00",$137.00,$108.00
4,jeanGrey,933616.0,179899.0,16868.0,1708.0,23420.0,4903.0,665.0,170.0,"$7,993.00","$1,679.00",$165.00,$108.00,"$68,590.00","$10,265.00",$822.00,$189.00


# Missing Data Observation

In [ ]:
#observe missing data
print('\nMissing values per column:')
print(df_untidy.isnull().sum())


Missing values per column:
Member                     0
TotalValue60s_heritage    16
TotalValue70s_heritage    10
TotalValue80s_heritage     3
TotalValue90s_heritage     4
TotalValue60s_ebay        16
TotalValue70s_ebay        10
TotalValue80s_ebay         3
TotalValue90s_ebay         4
TotalValue60s_wiz         16
TotalValue70s_wiz         10
TotalValue80s_wiz          3
TotalValue90s_wiz          4
TotalValue60s_oStreet     16
TotalValue70s_oStreet     10
TotalValue80s_oStreet      3
TotalValue90s_oStreet      4
dtype: int64


# Data Cleaning and Transformation
Performed through:
1. first melting the data into long format
2. splitting the variable column into "decade" and "market"
3. cleaning columns with str.replace()
4. finalize output

In [23]:
#melt
df_melted = df_untidy.melt(id_vars=['Member'], var_name='variable', value_name='TotalValue')
print('Shape after melting:', df_melted.shape)
print('\nSample of melted data:')
df_melted.head(10)

#split
split_cols = df_melted['variable'].str.split('_', expand=True)
df_melted['decade_raw'] = split_cols[0]
df_melted['Market']     = split_cols[1]
print('Unique decade_raw values:', df_melted['decade_raw'].unique())
print('Unique Market values:    ', df_melted['Market'].unique())

#clean
df_melted['Decade'] = df_melted['decade_raw'].str.replace('TotalValue', '', regex=False)
df_melted['TotalValue'] = (
    df_melted['TotalValue']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)

df_melted['TotalValue'] = pd.to_numeric(df_melted['TotalValue'], errors='coerce')

#finalize output
df_tidy = (
    df_melted[['Member', 'Decade', 'Market', 'TotalValue']]
    .dropna(subset=['TotalValue'])
    .reset_index(drop=True)
)

print('Final tidy shape:', df_tidy.shape)
print('\nFirst couple of rows of tidy data:')
df_tidy.head(10)

Shape after melting: (416, 3)

Sample of melted data:
Unique decade_raw values: ['TotalValue60s' 'TotalValue70s' 'TotalValue80s' 'TotalValue90s']
Unique Market values:     ['heritage' 'ebay' 'wiz' 'oStreet']
Final tidy shape: (284, 4)

First couple of rows of tidy data:


,Member,Decade,Market,TotalValue
0,warrenWorthington,60s,heritage,929056.0
1,hankMcCoy,60s,heritage,929776.0
2,scottSummers,60s,heritage,933616.0
3,bobbyDrake,60s,heritage,929776.0
4,jeanGrey,60s,heritage,933616.0
5,alexSummers,60s,heritage,34519.0
6,lornaDane,60s,heritage,76279.0
7,seanCassidy,60s,heritage,39200.0
8,ericMagnus,60s,heritage,582934.0
9,charlesXavier,60s,heritage,819537.0


# Visualizations: Pivot Table